# Profils des enquêtés

## Les pratiques de travail

In [1]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [3]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")

list_affil = pd.read_csv("../list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])].merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

In [4]:

df_col = pd.read_csv("../../le_questionnaire/dico_variable.csv", sep = ",")


In [5]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [6]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [7]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [8]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

In [9]:
df0[["q45_clé","q26_env_travail"]]

,q45_clé,q26_env_travail
0,PS8W-TJ9P,A mon domicile
1,GNEG-5QXQ,A mon domicile
2,FCCB-GMYA,A mon domicile;Dans mon bureau sur site
3,CV3G-95CT,Dans mon bureau sur site;A mon domicile
4,M5L8-UM9F,Dans mon bureau sur site;A mon domicile
...,...,...
114,MLSN-NBUS,"Autre, précisez;Dans mon bureau sur site;A mon..."
115,WAR9-LFJP,"Dans mon bureau sur site;A mon domicile;Autre,..."
116,HRZG-6JGE,"Autre, précisez;A mon domicile"
117,32TW-JLY5,A mon domicile;Dans mon bureau sur site


In [10]:
df_exp, gb_data = split_multiple_choices(df0[["q45_clé","q26_env_travail"]], "q26_env_travail", "q45_clé", sep = '|')
print(gb_data[["q26_env_travail","nb","freq","total"]].to_markdown(index=False))

| q26_env_travail          |   nb |    freq |   total |
|:-------------------------|-----:|--------:|--------:|
| A mon domicile           |  108 | 90.7563 |     119 |
| Dans mon bureau sur site |   70 | 58.8235 |     119 |
| Autre, précisez          |   31 | 26.0504 |     119 |


La première question interrogeant les pratiques de travail des enquêtés porte sur le lieu habituel de travail en classant par ordre de priorité les options suivantes :
- "À mon domicile"
- "Dans mon bureau sur site"
- "Autre"

90% des personnes interrogés disent travailler le plus souvent à leur domicile ({numref}`env_travail`). C'est aussi le lieu le plus souvent cité en premier. 71% des répondants travaillent ainsi prioritairement à leur domicile. Tandis que le bureau est en moyenne classé deuxième dans l'ordre des priorités.  

Parmis les autres lieux cités, on trouve tout d'abord les bibliothèques et les autres sites (CNRS, Campus Condorcet, partenaires de recherche). On note également que plusieurs répondants ont indiqué qu'ils travaillaient un peu partout, là où ils pouvaient faute de bureau disponible sur le campus. Ces réponses recoupent les résultats observés concernant les services à développer où la construction d'espace de travail avaient été choisi par plus de 80% des répondants ({numref}`11_service_to_develop`).


```{table} Où travaillez-vous le plus souvent ?
:name: env_travail


|   rang |   A mon domicile         |   Dans mon bureau sur site         |   Autre, précisez         |
|-------:|-------------------------:|-----------------------------------:|--------------------------:|
|      1 |               85 (71,4%) |                         29 (24,4%) |                 5  (4,2%) |
|      2 |               20 (16,8%) |                         36 (20,3%) |                12 (10,1%) |
|      3 |                3  (2,5%) |                          5  (4,2%) |                14 (11,8%) |
|Total   |              108 (90,8%) |                         70 (58,8%) |                31 (26,1%) |

```

In [11]:
#On compte le nombre de choix par personne

nb_choix = df_exp.groupby(["q45_clé"]).agg(nb=("q26_env_travail", "size")).reset_index()

np.mean(nb_choix.nb)
nb_choix.groupby(["nb"]).agg(freq=("q45_clé", "size")).reset_index()

,nb,freq
0,1,51
1,2,46
2,3,22


In [12]:
### Traitement des questions ordonnées : on calcule la fréquence par rang pour chaque option, puis le rang moyen
list_rank= []

for cle in df_exp.q45_clé.unique():
    dtmp = df_exp.loc[df_exp.q45_clé==cle]
    for n, r in enumerate(dtmp.q26_env_travail):
        dict_rank={"q45_clé":cle,
                   "q26_env_travail":r,
                   "rang":n+1}
        list_rank.append(dict_rank)
list_rank
    
df_rank = pd.DataFrame.from_dict(list_rank)  

df_rank

,q45_clé,q26_env_travail,rang
0,PS8W-TJ9P,A mon domicile,1
1,GNEG-5QXQ,A mon domicile,1
2,FCCB-GMYA,A mon domicile,1
3,FCCB-GMYA,Dans mon bureau sur site,2
4,CV3G-95CT,Dans mon bureau sur site,1
...,...,...,...
204,HRZG-6JGE,A mon domicile,2
205,32TW-JLY5,A mon domicile,1
206,32TW-JLY5,Dans mon bureau sur site,2
207,6GVE-N5NJ,A mon domicile,1


In [13]:
d_class = df_rank.groupby(["rang","q26_env_travail"]).agg(nb=("q45_clé","size")).reset_index()
d_class["freq"] = round(d_class.nb/119*100, 1)
d_class_tab = pd.pivot(d_class, index= "rang", columns="q26_env_travail", values="freq").reset_index()
print(d_class_tab[["rang","A mon domicile",  "Dans mon bureau sur site",  "Autre, précisez"]].to_markdown(index=False))
#df_rank.groupby(["q26_env_travail","rang"]).agg(nb=("q45_clé","size"))

|   rang |   A mon domicile |   Dans mon bureau sur site |   Autre, précisez |
|-------:|-----------------:|---------------------------:|------------------:|
|      1 |             71.4 |                       24.4 |               4.2 |
|      2 |             16.8 |                       30.3 |              10.1 |
|      3 |              2.5 |                        4.2 |              11.8 |
